# Preparación de Datos para TP2 (Versión Final con los 3 Archivos)

### Objetivo
Este notebook consolida todo el proceso de carga, limpieza y fusión de datos. Se cargan los datos horarios, los datos de estaciones y las estadísticas históricas para generar un único DataFrame enriquecido y listo para el análisis de la Parte 2. Se conserva la columna `ESTACION` en el resultado final.

In [59]:
import pandas as pd
import numpy as np
import re

## 1. Carga de Datos

Se cargan los tres archivos de datos: horarios, de estaciones y de estadísticas normales, usando el método `read_fwf` con los parámetros que demostraron ser correctos para interpretar el formato de estos archivos.

In [91]:
# --- Celda de Carga de Datos (Versión Definitiva y Funcional) ---

# NOTA: Si da 'FileNotFoundError', cambia la siguiente línea a path = 'SMN_data/'
path = 'SMN_data/'

# --- 1. Carga de datos horarios (Este método funciona) ---
df_hor = pd.read_fwf(
    path + "datohorario20250715.txt",
    encoding='latin1', sep='\t', skiprows=2,
    names=["FECHA", "HORA", "TEMP", "HUM", "PNM", "DD", "FF", "ESTACION"]
)

# --- 2. Carga de datos de estaciones (Este método funciona) ---
station_col_specs = [(0, 40), (40, 75), (75, 82), (82, 89), (89, 96), (96, 103), (103, 110), (110, 117), (117, 125)]
station_col_names = ['ESTACION', 'PROVINCIA', 'LAT_GRAD', 'LAT_MIN', 'LON_GRAD', 'LON_MIN', 'ALTURA', 'NUM', 'NroOACI']
df_est = pd.read_fwf(
    path + "estaciones_smn.txt",
    colspecs=station_col_specs, names=station_col_names,
    skiprows=2, encoding='latin1'
)

# --- 3. Carga de estadísticas (CORRECCIÓN DEFINITIVA con pd.read_csv) ---
# Usamos la función estándar para archivos separados por tabulaciones.
df_stats = pd.read_csv(
    path + "Estadísticas normales Datos abiertos 1991-2020.txt",
    sep='\t',
    skiprows=7,
    encoding='cp1252',
    names=["ESTACION", "Valor Medio de", "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
)

print("Carga de los 3 archivos con métodos específicos y correctos completada.")

Carga de los 3 archivos con métodos específicos y correctos completada.


In [92]:
# --- Celda de Inspección de Datos ---

print("--- 1. Revisión de Datos Horarios (df_hor) ---")
print(f"Forma: {df_hor.shape}")
print("Verifica que las columnas (TEMP, HUM, etc.) tengan datos.")
display(df_hor.head())

print("\n" + "="*80 + "\n")

print("--- 2. Revisión de Datos de Estaciones (df_est) ---")
print(f"Forma: {df_est.shape}")
print("Verifica que las columnas (PROVINCIA, LAT, etc.) se hayan separado correctamente.")
display(df_est.head())

print("\n" + "="*80 + "\n")

print("--- 3. Revisión de Estadísticas Históricas (df_stats) ---")
print(f"Forma: {df_stats.shape}")
print("Verifica que la columna 'Valor Medio de' y los meses contengan datos.")
display(df_stats.head())

--- 1. Revisión de Datos Horarios (df_hor) ---
Forma: (2074, 8)
Verifica que las columnas (TEMP, HUM, etc.) tengan datos.


,FECHA,HORA,TEMP,HUM,PNM,DD,FF,ESTACION
0,15072025.0,0.0,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO
1,15072025.0,1.0,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO
2,15072025.0,2.0,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO
3,15072025.0,3.0,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO
4,15072025.0,4.0,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO




--- 2. Revisión de Datos de Estaciones (df_est) ---
Forma: (120, 9)
Verifica que las columnas (PROVINCIA, LAT, etc.) se hayan separado correctamente.


,ESTACION,PROVINCIA,LAT_GRAD,LAT_MIN,LON_GRAD,LON_MIN,ALTURA,NUM,NroOACI
0,BASE BELGRANO II ANTARTIDA,-77,52.0,-34.0,3.0,7.0,256.0,89034 S,AYB
1,BASE CARLINI (EX JUBANY) ANTARTIDA,-62,14.0,-58.0,3.0,9.0,11.0,89053 S,AYJ
2,BASE ESPERANZA ANTARTIDA,-63,23.0,-56.0,5.0,9.0,24.0,88963 S,AYE
3,BASE MARAMBIO ANTARTIDA,-64,14.0,-56.0,3.0,7.0,198.0,89055 S,AWB
4,BASE ORCADAS ANTARTIDA,-60,44.0,-44.0,4.0,4.0,12.0,88968 S,AYO




--- 3. Revisión de Estadísticas Históricas (df_stats) ---
Forma: (785, 14)
Verifica que la columna 'Valor Medio de' y los meses contengan datos.


,ESTACION,Valor Medio de,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic
Estación,Valor Medio de,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic,NaN
LA QUIACA OBSERVATORIO,Temperatura (°C),13.2,13.0,12.8,11.3,7.3,4.8,4.5,7.0,10.0,12.4,13.4,13.9,NaN
LA QUIACA OBSERVATORIO,Temperatura máxima (°C),20.6,20.4,20.6,20.3,17.8,16.3,16.1,18.0,20.0,21.7,22.5,22.2,NaN
LA QUIACA OBSERVATORIO,Temperatura mínima (°C),7.7,7.6,6.6,3.1,-2.5,-5.7,-6.2,-4.0,-0.4,3.3,5.5,7.3,NaN
LA QUIACA OBSERVATORIO,Humedad relativa (%),62.6,63.2,60.3,46.0,32.6,27.4,25.7,26.7,32.1,42.4,48.6,55.8,NaN


## 2. Limpieza y Preprocesamiento

Se aplican los pasos de limpieza a cada DataFrame para asegurar la calidad y compatibilidad de los datos antes de las fusiones.

In [95]:
# --- Celda de Limpieza (Corregida) ---

# --- Limpieza de df_hor ---
df_hor['FECHA'] = pd.to_datetime(df_hor['FECHA'], format='%d%m%Y', errors='coerce')
numeric_cols_hor = ["HORA", "TEMP", "HUM", "PNM", "DD", "FF"]
for col in numeric_cols_hor:
    df_hor[col] = pd.to_numeric(df_hor[col], errors='coerce')
cols_to_check = [col for col in df_hor.columns if col != 'ESTACION']
df_hor.dropna(subset=cols_to_check, how='all', inplace=True)

# --- Limpieza de df_est ---
df_est['LAT'] = df_est['LAT_GRAD'] + (df_est['LAT_MIN'] / 60)
df_est['LON'] = -1 * (abs(df_est['LON_GRAD']) + (df_est['LON_MIN'] / 60))

# --- Limpieza y Transformación de df_stats ---
df_stats['Valor Medio de'] = df_stats['Valor Medio de'].str.strip()
df_stats_temp = df_stats[df_stats['Valor Medio de'] == 'Temperatura (°C)'].copy()
df_stats_long = df_stats_temp.melt(
    id_vars=['ESTACION'],
    value_vars=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    var_name='MES_NOMBRE', value_name='TEMP_MEDIA_HIST'
)
meses_map = {'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Ago': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12}
df_stats_long['MES'] = df_stats_long['MES_NOMBRE'].map(meses_map)
df_stats_long['TEMP_MEDIA_HIST'] = pd.to_numeric(df_stats_long['TEMP_MEDIA_HIST'], errors='coerce')
df_stats_final = df_stats_long[['ESTACION', 'MES', 'TEMP_MEDIA_HIST']]

print("Limpieza y preprocesamiento de los 3 archivos completada.")

Limpieza y preprocesamiento de los 3 archivos completada.


## 3. Fusión Secuencial de los DataFrames

Se define una única función de limpieza para los nombres de las estaciones y se aplica a los tres DataFrames. Luego, se realizan las fusiones en secuencia.

In [96]:
# --- Celda de Fusión (Reestructurada para Mayor Robustez) ---

# --- 1. Preparación de los 3 DataFrames ---

# Función de limpieza definitiva para estandarizar nombres
def limpiar_nombre(nombre):
    nombre_limpio = str(nombre).upper().strip()
    # Eliminar sufijos comunes y contenido entre paréntesis
    nombre_limpio = re.sub(r' AERO$', '', nombre_limpio)
    nombre_limpio = re.sub(r' B\.A\.$', '', nombre_limpio)
    nombre_limpio = re.sub(r'\(.*?\)', '', nombre_limpio)
    # Eliminar posibles sufijos de provincia de 2 letras
    nombre_limpio = re.sub(r'\s[A-Z]{2}$', '', nombre_limpio)
    # Limpieza general final
    nombre_limpio = re.sub(r'\s+', ' ', nombre_limpio).strip()
    return nombre_limpio

# Preparamos el DataFrame de estadísticas
df_stats_temp = df_stats[df_stats['Valor Medio de'] == 'Temperatura (°C)'].copy()
df_stats_long = df_stats_temp.melt(
    id_vars=['ESTACION'],
    value_vars=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    var_name='MES_NOMBRE', value_name='TEMP_MEDIA_HIST'
)
meses_map = {'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Ago': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12}
df_stats_long['MES'] = df_stats_long['MES_NOMBRE'].map(meses_map)
df_stats_long['TEMP_MEDIA_HIST'] = pd.to_numeric(df_stats_long['TEMP_MEDIA_HIST'], errors='coerce')
df_stats_final = df_stats_long[['ESTACION', 'MES', 'TEMP_MEDIA_HIST']]

# Creamos la clave de unión estandarizada en los 3 DataFrames
df_hor['ESTACION_KEY'] = df_hor['ESTACION'].apply(limpiar_nombre)
df_est['ESTACION_KEY'] = df_est['ESTACION'].apply(limpiar_nombre)
df_stats_final['ESTACION_KEY'] = df_stats_final['ESTACION'].apply(limpiar_nombre)


# --- 2. Primera Fusión: Crear la "Tabla Maestra" de Estaciones ---
# Unimos la información geográfica (df_est) con la histórica (df_stats_final)
df_maestro_estaciones = pd.merge(
    df_est,
    df_stats_final,
    on='ESTACION_KEY',
    how='outer', # Usamos 'outer' para ver todas las estaciones de ambos archivos
    suffixes=('_geo', '_hist')
)

print("--- Diagnóstico de la Tabla Maestra de Estaciones ---")
# Verificamos si hay estaciones que solo existen en un archivo
# (Esto nos ayuda a ver si la limpieza de nombres fue exitosa)
print(df_maestro_estaciones[['ESTACION_geo', 'ESTACION_hist']].isnull().sum())
print("-" * 50)


# --- 3. Fusión Final: Unir Datos Horarios con la Tabla Maestra ---
df_combinado = df_hor.copy()
df_combinado['MES'] = df_combinado['FECHA'].dt.month

df_enriquecido = pd.merge(
    df_combinado,
    df_maestro_estaciones,
    on=['ESTACION_KEY', 'MES'],
    how='left'
)

# Rellenamos nulos que puedan quedar
df_enriquecido['TEMP_MEDIA_HIST'] = df_enriquecido['TEMP_MEDIA_HIST'].fillna(df_enriquecido['TEMP_MEDIA_HIST'].mean())

print("Fusiones completadas.")
display(df_enriquecido.head())

--- Diagnóstico de la Tabla Maestra de Estaciones ---
ESTACION_geo       0
ESTACION_hist    120
dtype: int64
--------------------------------------------------
Fusiones completadas.


,FECHA,HORA,TEMP,HUM,PNM,DD,FF,ESTACION,ESTACION_KEY,MES,...,LAT_MIN,LON_GRAD,LON_MIN,ALTURA,NUM,NroOACI,LAT,LON,ESTACION_hist,TEMP_MEDIA_HIST
0,2025-07-15,0.0,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO,AEROPARQUE,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-07-15,1.0,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO,AEROPARQUE,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-07-15,2.0,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO,AEROPARQUE,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-07-15,3.0,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO,AEROPARQUE,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-07-15,4.0,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO,AEROPARQUE,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
# --- Celda de Diagnóstico (Corregida para inspeccionar valores) ---

print("--- Iniciando Diagnóstico de Fusión de Datos Históricos ---")

# PASO CLAVE: Verificamos qué valores existen realmente en la columna del archivo de estadísticas
print("\nValores disponibles en la columna 'Valor Medio de' del archivo de estadísticas:")
valores_disponibles = df_stats['Valor Medio de'].dropna().unique()
print(valores_disponibles)
print("-" * 50)
print("NOTA: Copia el valor de temperatura exacto que aparece en la lista de arriba para usarlo en la siguiente celda.")
print("-" * 50)


# 1. Obtenemos las listas de nombres de estaciones únicas de ambas fuentes
# (El resto del código de diagnóstico se ejecutará para ser completo, aunque la causa principal ya fue encontrada)
try:
    estaciones_combinado = set(df_combinado['ESTACION'].dropna().unique())
    estaciones_stats = set(df_stats_final['ESTACION'].dropna().unique())

    # 2. Encontramos las estaciones que están en tus datos pero no en el archivo de estadísticas
    unmatched_keys = sorted(list(estaciones_combinado - estaciones_stats))

    if not unmatched_keys:
        print("\n✅ ¡Buenas noticias! Todas las estaciones coinciden con el archivo de estadísticas históricas.")
    else:
        print(f"\n🚨 Se encontraron {len(unmatched_keys)} estaciones que no tienen correspondencia en los datos históricos.")
        # ... (el resto del código de sugerencias)

except NameError:
    print("\nEl diagnóstico de comparación no puede continuar porque 'df_stats_final' está vacío.")
    print("Por favor, corrige el nombre del valor de temperatura en la celda de fusión y vuelve a ejecutar todo.")


print("\n--- Fin del Diagnóstico ---")

--- Iniciando Diagnóstico de Fusión de Datos Históricos ---

Valores disponibles en la columna 'Valor Medio de' del archivo de estadísticas:
[]
--------------------------------------------------
NOTA: Copia el valor de temperatura exacto que aparece en la lista de arriba para usarlo en la siguiente celda.
--------------------------------------------------

🚨 Se encontraron 118 estaciones que no tienen correspondencia en los datos históricos.

--- Fin del Diagnóstico ---


## 4. Feature Engineering y Selección Final

Finalmente, creamos las características finales y seleccionamos las columnas para el DataFrame de salida.

In [ ]:
# Crear features cíclicas para la hora
df_enriquecido['HORA_sen'] = np.sin(2 * np.pi * df_enriquecido['HORA']/24.0)
df_enriquecido['HORA_cos'] = np.cos(2 * np.pi * df_enriquecido['HORA']/24.0)

# Convertir la columna PROVINCIA en variables dummy
df_final = pd.get_dummies(df_enriquecido, columns=['PROVINCIA'], prefix='PROV', drop_first=True)

# Columnas a eliminar del set final
cols_to_drop = [
    "FECHA", "HORA", "ESTACION_KEY", "MES",
    "ESTACION_est", "LAT_GRAD", "LAT_MIN", 
    "LON_GRAD", "LON_MIN", "NUM", "NroOACI"
]

# Creamos el DataFrame final, conservando la columna 'ESTACION' original
df_para_tp2 = df_final.drop(columns=cols_to_drop, errors='ignore')

# Guardamos el resultado en un nuevo archivo CSV
df_para_tp2.to_csv("df_para_tp2.csv", index=False)

print("DataFrame final enriquecido creado y guardado como 'df_para_tp2.csv'")
display(df_para_tp2.head())

DataFrame final enriquecido creado y guardado como 'df_para_tp2.csv'


,TEMP,HUM,PNM,DD,FF,ESTACION,ALTURA,LAT,LON,TEMP_MEDIA_HIST,...,PROV_MISIONES,PROV_NEUQUEN,PROV_RIO NEGRO,PROV_SALTA,PROV_SAN JUAN,PROV_SAN LUIS,PROV_SANTA CRUZ,PROV_SANTA FE,PROV_TIERRA DEL FUEG,PROV_TUCUMAN
0,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO,6.0,-33.45,-58.416667,NaN,...,False,False,False,False,False,False,False,False,False,False
1,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO,6.0,-33.45,-58.416667,NaN,...,False,False,False,False,False,False,False,False,False,False
2,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO,6.0,-33.45,-58.416667,NaN,...,False,False,False,False,False,False,False,False,False,False
3,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO,6.0,-33.45,-58.416667,NaN,...,False,False,False,False,False,False,False,False,False,False
4,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO,6.0,-33.45,-58.416667,NaN,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# --- Celda de Diagnóstico para la Fusión de Datos Históricos ---

print("--- Iniciando Diagnóstico de Fusión de Datos Históricos ---")

# 1. Preparamos el DataFrame de estadísticas (repitiendo la lógica de limpieza)
df_stats_temp_diag = df_stats[df_stats['Valor Medio de'] == 'Temperatura (°C)'].copy()
df_stats_long_diag = df_stats_temp_diag.melt(
    id_vars=['ESTACION'],
    value_vars=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    var_name='MES_NOMBRE', value_name='TEMP_MEDIA_HIST'
)
df_stats_final_diag = df_stats_long_diag[['ESTACION']]


# 2. Obtenemos las listas de nombres de estaciones únicas de ambas fuentes
#    Usamos una limpieza simple (quitar espacios y mayúsculas) para una primera comparación
estaciones_combinado = set(df_combinado['ESTACION'].str.strip().str.upper().dropna().unique())
estaciones_stats = set(df_stats_final_diag['ESTACION'].str.strip().str.upper().dropna().unique())

# 3. Encontramos las estaciones que no coinciden
unmatched_keys = sorted(list(estaciones_combinado - estaciones_stats))

if not unmatched_keys:
    print("\n✅ ¡Buenas noticias! Todos los nombres de estaciones parecen coincidir. El problema puede ser otro.")
else:
    print(f"\n🚨 Se encontraron {len(unmatched_keys)} estaciones que no tienen correspondencia en los datos históricos.")
    print("   Esto está causando que los valores de 'TEMP_MEDIA_HIST' no se asignen.")
    print("\n   A continuación se muestran las estaciones problemáticas y sus posibles coincidencias:")

    from thefuzz import process
    
    suggestions = []
    for key in unmatched_keys:
        best_match, score = process.extractOne(key, list(estaciones_stats))
        suggestions.append({
            'Estacion en tus Datos (df_combinado)': key,
            'Sugerencia en Archivo de Estadísticas': best_match,
            'Puntaje (0-100)': score
        })

    suggestions_df = pd.DataFrame(suggestions)
    display(suggestions_df)

print("\n--- Fin del Diagnóstico ---")

--- Iniciando Diagnóstico de Fusión de Datos Históricos ---

🚨 Se encontraron 118 estaciones que no tienen correspondencia en los datos históricos.
   Esto está causando que los valores de 'TEMP_MEDIA_HIST' no se asignen.

   A continuación se muestran las estaciones problemáticas y sus posibles coincidencias:


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# --- Celda de Diagnóstico (Corregida para inspeccionar valores) ---

print("--- Iniciando Diagnóstico de Fusión de Datos Históricos ---")

# PASO CLAVE: Verificamos qué valores existen en la columna 'Valor Medio de'
print("\nValores disponibles en la columna 'Valor Medio de' del archivo de estadísticas:")
valores_disponibles = df_stats['Valor Medio de'].dropna().unique()
print(valores_disponibles)
print("-" * 50)
print("NOTA: Copia el valor de temperatura exacto de la lista de arriba para usarlo en la siguiente celda.")
print("-" * 50)

# El resto del código se salta si la lista de estadísticas está vacía, previniendo el error.
if 'df_stats_final' in locals() and not df_stats_final.empty:
    # ... (código de comparación omitido para claridad)
    pass
else:
    print("\nEl diagnóstico de comparación no puede continuar porque los datos de estadísticas no se cargaron.")
    print("Por favor, corrige el texto de la temperatura en la celda de fusión y vuelve a ejecutar.")

print("\n--- Fin del Diagnóstico ---")

--- Iniciando Diagnóstico de Fusión de Datos Históricos ---

Valores disponibles en la columna 'Valor Medio de' del archivo de estadísticas:
[]
--------------------------------------------------
NOTA: Copia el valor de temperatura exacto de la lista de arriba para usarlo en la siguiente celda.
--------------------------------------------------

El diagnóstico de comparación no puede continuar porque los datos de estadísticas no se cargaron.
Por favor, corrige el texto de la temperatura en la celda de fusión y vuelve a ejecutar.

--- Fin del Diagnóstico ---
